# Feast RayJob CR Execution on Existing Ray Cluster

This notebook demonstrates how to use Feast with RayJob CRs submitted to an existing Ray cluster via CodeFlare SDK.

## Overview

This approach provides:
- **Efficient cluster reuse**: Existing Ray cluster stays alive during DAG execution
- **Proper RayJob CR management**: Uses CodeFlare SDK for RayJob CR lifecycle
- **Step isolation**: Each DAG step runs as separate RayJob CR
- **Fast execution**: No cluster startup overhead
- **Resource optimization**: Cluster remains available for other workloads


## Prerequisites

Before running this notebook, ensure you have:

1. **CodeFlare SDK installed**: `pip install codeflare-sdk`
2. **KubeRay operator** installed in your Kubernetes cluster
3. **Existing Ray cluster** running in Kubernetes
4. **kubectl** configured to access your cluster
5. **Feast** installed with Ray support


In [ ]:
# Install required dependencies
%pip install feast codeflare-sdk ray[data] pandas pyarrow dill


## 1. Setup and Configuration

First, let's configure Feast to use the existing cluster + RayJob CR approach.


In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

# Add Feast to Python path (adjust path as needed)
feast_path = Path.cwd() / "sdk" / "python"
if feast_path.exists():
    sys.path.insert(0, str(feast_path))

from feast import FeatureStore, Entity, FeatureView, Field
from feast.data_source import FileSource
from feast.infra.compute_engines.ray.config import RayComputeEngineConfig
from feast.infra.compute_engines.ray.codeflare_orchestrator import CodeFlareDAGOrchestrator

print("✅ Imports successful")


### 1.1 Create Feature Store Configuration

Let's create a `feature_store.yaml` configuration that uses the existing cluster + RayJob CR approach.


In [ ]:
# Create feature store configuration
feature_store_config = """
project: feast_rayjob_demo
registry: data/registry.db
provider: local

# Offline store configuration
offline_store:
  type: file
  path: data/offline_store

# Ray compute engine configuration for existing cluster + RayJob CRs
batch_engine:
  type: rayjob.engine                    # Use RayJob CR-based execution
  max_workers: 5                         # Maximum number of workers per step
  enable_optimization: true              # Enable performance optimizations
  broadcast_join_threshold_mb: 50        # Broadcast join threshold (MB)
  target_partition_size_mb: 32          # Target partition size (MB)
  window_size_for_joins: "1H"           # Time window for distributed joins
  
  # CodeFlare SDK configuration for existing cluster + RayJob CRs
  namespace: default                     # Kubernetes namespace for RayJob CRs
  step_timeout_seconds: 1800             # Timeout for individual DAG steps (30 min)
  use_codeflare_sdk: true               # Use CodeFlare SDK for RayJob management
  use_existing_cluster: true            # Use existing Ray cluster
  existing_cluster_name: my-ray-cluster # Name of existing Ray cluster
  cluster_lifecycle_mode: per_job       # Not used with existing cluster

# Online store for serving features
online_store:
  path: data/online_store.db

entity_key_serialization_version: 3
"""

# Write configuration to file
with open("feature_store.yaml", "w") as f:
    f.write(feature_store_config)

print("✅ Feature store configuration created")
print("📋 Configuration highlights:")
print("   • Engine type: rayjob.engine")
print("   • CodeFlare SDK: enabled")
print("   • Existing cluster: my-ray-cluster")
print("   • Step timeout: 30 minutes")
print("   • CodeFlare SDK handles: image, resources, monitoring automatically")


### 1.2 Verify CodeFlare SDK Installation

Let's check if CodeFlare SDK is properly installed and can connect to your cluster.


In [ ]:
try:
    import codeflare_sdk
    print("✅ CodeFlare SDK is installed")
    print(f"   Version: {codeflare_sdk.__version__ if hasattr(codeflare_sdk, '__version__') else 'Unknown'}")
except ImportError:
    print("❌ CodeFlare SDK not found")
    print("   Please install with: pip install codeflare-sdk")
    raise

# Check if we can access Kubernetes
try:
    import subprocess
    result = subprocess.run(["kubectl", "get", "nodes"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ kubectl is configured and can access cluster")
    else:
        print("❌ kubectl not configured or cluster not accessible")
        print(f"   Error: {result.stderr}")
except FileNotFoundError:
    print("❌ kubectl not found")
    print("   Please install kubectl and configure it to access your cluster")


## 2. Create Sample Data and Feature Definitions

Let's create some sample data and define features to demonstrate the RayJob CR execution.


In [ ]:
# Create directories
os.makedirs("data", exist_ok=True)
os.makedirs("data/offline_store", exist_ok=True)

# Generate sample data
np.random.seed(42)
n_samples = 1000

# Create entity data
entity_data = {
    "user_id": np.random.randint(1, 100, n_samples),
    "event_timestamp": pd.date_range(
        start=datetime.now() - timedelta(days=30),
        end=datetime.now(),
        periods=n_samples
    ),
    "amount": np.random.uniform(10, 1000, n_samples),
    "category": np.random.choice(["food", "transport", "entertainment", "shopping"], n_samples),
    "rating": np.random.uniform(1, 5, n_samples),
}

df = pd.DataFrame(entity_data)
df.to_parquet("data/offline_store/transactions.parquet", index=False)

print("✅ Sample data created")
print(f"   Records: {len(df)}")
print(f"   Columns: {list(df.columns)}")
print(f"   Date range: {df['event_timestamp'].min()} to {df['event_timestamp'].max()}")
print("\n📊 Sample data:")
print(df.head())


In [ ]:
# Define Feast entities and features
from feast.types import Float64, Int64, String

# Define entity
user_entity = Entity(
    name="user_id",
    description="User identifier",
    join_keys=["user_id"]
)

# Define data source
transaction_source = FileSource(
    name="transaction_source",
    path="data/offline_store/transactions.parquet",
    timestamp_field="event_timestamp",
)

# Define feature view
transaction_features = FeatureView(
    name="transaction_features",
    description="Transaction features",
    entities=[user_entity],
    ttl=timedelta(days=1),
    schema=[
        Field(name="amount", dtype=Float64),
        Field(name="category", dtype=String),
        Field(name="rating", dtype=Float64),
    ],
    source=transaction_source,
)

print("✅ Feast entities and features defined")
print(f"   Entity: {user_entity.name}")
print(f"   Feature view: {transaction_features.name}")
print(f"   Features: {[field.name for field in transaction_features.schema]}")


## 3. Initialize Feature Store and Demonstrate RayJob CR Execution

Now let's initialize the Feature Store and demonstrate the RayJob CR execution.


In [ ]:
# Initialize Feature Store
store = FeatureStore(repo_path=".")

# Apply the feature definitions
store.apply([user_entity, transaction_features])

print("✅ Feature Store initialized")
print(f"   Project: {store.project}")
print(f"   Registry: {store.registry}")
print(f"   Batch engine: {store.repo_config.batch_engine.type}")
print(f"   CodeFlare SDK: {store.repo_config.batch_engine.use_codeflare_sdk}")
print(f"   Existing cluster: {store.repo_config.batch_engine.existing_cluster_name}")


In [ ]:
# Create entity dataframe for historical feature retrieval
entity_df = pd.DataFrame({
    "user_id": [1, 2, 3, 4, 5],
    "event_timestamp": [datetime.now() - timedelta(days=1)] * 5
})

print("📋 Entity DataFrame for historical feature retrieval:")
print(entity_df)
print(f"\n🎯 Features to retrieve: {[f'{transaction_features.name}:{field.name}' for field in transaction_features.schema]}")


In [ ]:
# Get historical features - this will trigger RayJob CR execution
print("🚀 Starting historical feature retrieval...")
print("   This will submit RayJob CRs to the existing Ray cluster")
print("   Each DAG step will run as a separate RayJob CR")

try:
    # This is where the magic happens - RayJob CR execution!
    features = store.get_historical_features(
        entity_df=entity_df,
        features=[
            f"{transaction_features.name}:amount",
            f"{transaction_features.name}:category",
            f"{transaction_features.name}:rating",
        ]
    ).to_df()
    
    print("✅ Historical feature retrieval completed successfully!")
    print(f"   Retrieved {len(features)} feature records")
    print("\n📊 Retrieved features:")
    print(features)
    
except Exception as e:
    print(f"❌ Historical feature retrieval failed: {e}")
    print("\n🔍 Troubleshooting tips:")
    print("   1. Check if Ray cluster is running: kubectl get rayclusters")
    print("   2. Check RayJob CRs: kubectl get rayjobs")
    print("   3. Check pod logs: kubectl logs <rayjob-pod-name>")
    print("   4. Verify CodeFlare SDK installation: pip install codeflare-sdk")


## 4. Monitor RayJob CR Execution

Let's check the RayJob CRs that were created during the execution.


In [ ]:
# Check RayJob CRs in the cluster
def check_rayjobs(namespace="default"):
    """Check RayJob CRs in the namespace."""
    try:
        import subprocess
        result = subprocess.run(
            ["kubectl", "get", "rayjobs", "-n", namespace, "-o", "wide"],
            capture_output=True, text=True
        )
        
        if result.returncode == 0:
            print("📋 RayJob CRs in cluster:")
            print(result.stdout)
        else:
            print(f"❌ Error getting RayJob CRs: {result.stderr}")
            
    except Exception as e:
        print(f"❌ Error checking RayJob CRs: {e}")

# Check RayJob CRs
check_rayjobs()


## 5. Execution Summary and Benefits

Let's summarize what happened and the benefits of this approach.


In [ ]:
print("🎯 Execution Summary")
print("=" * 50)
print("\n✅ What happened:")
print("   1. Feast connected to existing Ray cluster: my-ray-cluster")
print("   2. DAG steps were serialized into discrete RayJob CRs")
print("   3. RayJob CRs were submitted to existing cluster via CodeFlare SDK")
print("   4. Each DAG step executed as separate RayJob CR")
print("   5. Steps waited for dependencies to complete")
print("   6. Results were collected and returned")
print("   7. Cluster remained alive for future use")

print("\n📊 Benefits achieved:")
print("   • Efficient cluster reuse (no startup overhead)")
print("   • Proper RayJob CR management via CodeFlare SDK")
print("   • Step isolation within RayJob CRs")
print("   • Automatic dependency resolution")
print("   • Resource optimization")

print("\n🔍 Monitoring Commands:")
print("   # Check RayJob CRs:")
print("   kubectl get rayjobs -n default")
print("   ")
print("   # Check Ray cluster:")
print("   kubectl get rayclusters -n default")
print("   ")
print("   # Check pods:")
print("   kubectl get pods -l ray.io/cluster-name=my-ray-cluster -n default")
print("   ")
print("   # View RayJob CR logs:")
print("   kubectl logs <rayjob-pod-name> -n default")
print("   ")
print("   # Describe RayJob CR:")
print("   kubectl describe rayjob <rayjob-name> -n default")


## Conclusion

This notebook demonstrated how to use Feast with RayJob CRs submitted to an existing Ray cluster via CodeFlare SDK. This approach provides:

- **Efficient cluster reuse** - No startup overhead
- **Proper RayJob CR management** - Via CodeFlare SDK
- **Step isolation** - Each DAG step runs as separate RayJob CR
- **Resource optimization** - Cluster remains available for other workloads
- **Automatic dependency management** - Steps wait for dependencies

The implementation successfully bridges the gap between Feast's DAG execution and Kubernetes-native RayJob CR management, providing a scalable and efficient solution for feature engineering workloads.
